# CLI Agent Notebook

This notebook demonstrates an agent-friendly workflow for the `ct` CLI. Each executable cell runs one command and keeps the decision boundary clear: use text reports for reading and JSON data for exact queries.

Source session:

```text
019e8d04-1f38-79f3-b5a8-2e36a86be145
```


## Rule Of Thumb

Report commands default to markdown for reading. Use `--output json` when querying exact fields with `jq` or scripts. Detail commands already default to compact JSON.


In [ ]:
%%bash
SESSION=019e8d04-1f38-79f3-b5a8-2e36a86be145
echo "$SESSION"

## Cell 1: Orient On The Last Turn

Read only the latest visible turn. This is the cheapest first pass when an agent needs to understand what happened recently. Overview prints short ids for reading; use `--output json` when a later command needs the full UUID.


In [ ]:
%%bash
SESSION=019e8d04-1f38-79f3-b5a8-2e36a86be145
ct session overview "$SESSION" --turns 1

Useful signal from this source session:

```text
`- turn 9b964295  completed  [97fa232b,ce39ebd2,38a4b92b,c36f83b6+28]  1. for ct session usage : remove the activity property...
   +- assistant: I’ll adjust the last refactor rather than layering another surface on top of it...
   +- SearchText: 'json|JSON|format.*json|session stats|session usage|activity|cost_drive...
   `- ... 37 more activities
```

Decision: the last turn is the current implementation turn. Use `--output json` to recover full step ids for detail drill-down instead of reading the whole raw log.


## Cell 2: Find The Expensive Turn

Use JSON when selecting a turn by exact metrics.

In [ ]:
%%bash
SESSION=019e8d04-1f38-79f3-b5a8-2e36a86be145
ct session usage "$SESSION" --output json \
  | jq '.turns | max_by(.usage.cost) | {
      turn_id: .id,
      cost_usd: .usage.cost,
      input: .usage.in,
      cached_input: .usage.cache,
      output: .usage.out,
      total: .usage.total,
      activity_usage: .act
    }'


Expected signal from this source session:

```json
{
  "turn_id": "9b964295-8dfd-5849-b028-13651a25de3d",
  "cost_usd": 1.936248,
  "input": 2195724,
  "cached_input": 2087296,
  "output": 11682,
  "total": 2207406,
  "activity_usage": [
    {"kind": "tool_steps", "usage": {"cost": 1.457921}},
    {"kind": "response_steps", "usage": {"cost": 0.478327}}
  ]
}
```

Decision: turn `9b964295-8dfd-5849-b028-13651a25de3d` dominates cost, so inspect that turn before spending tokens on earlier turns.


## Cell 3: Read Turn Usage

Use the default text report when an agent needs to read the usage accounting without writing a query.


In [ ]:
%%bash
SESSION=019e8d04-1f38-79f3-b5a8-2e36a86be145
ct session usage "$SESSION" \
  --turn 9b964295-8dfd-5849-b028-13651a25de3d


Useful fields in `--output json`:

```json
{
  "id": "9b964295-8dfd-5849-b028-13651a25de3d",
  "usage": {
    "in": 2195724,
    "cache": 2087296,
    "out": 11682,
    "cost": 1.936248
  },
  "act": [
    {
      "kind": "tool_steps",
      "usage": {"cost": 1.457921}
    },
    {
      "kind": "response_steps",
      "usage": {"cost": 0.478327}
    }
  ]
}
```

Decision: the turn is expensive mostly because of tool steps, but the command still avoids expanding paths, commands, and raw tool output. Use step detail for evidence.


## Cell 4: Drill Into A Specific Step

Use the first implementation step from the overview. Detail commands take full UUIDs, so retrieve the full id from `overview --output json` or a saved JSON artifact.


In [ ]:
%%bash
ct session step-detail 97fa232b-2f52-5ddf-b38c-d82ea94e2ac2

Expected signal:

```json
[
  {
    "id": "97fa232b-2f52-5ddf-b38c-d82ea94e2ac2",
    "type": "assistant_response",
    "ops": ["text_reply"],
    "shape": {
      "texts": [
        "I’ll adjust the last refactor rather than layering another surface on top of it..."
      ]
    },
    "events": ["0a16721d-6f97-58ac-bfd2-c17049865103"]
  }
]
```

Decision: this step is an assistant planning response. Continue through the step ids when looking for edits, commands, or verification evidence.

## Cell 5: Save A Stable JSON Artifact

Use `--output json` when you need a machine-readable payload on stdout.


In [ ]:
%%bash
SESSION=019e8d04-1f38-79f3-b5a8-2e36a86be145
ct session overview "$SESSION" --turns 1 --output json \
  | jq '.sessions[0].turns[0].steps | length'

Expected result for this source session:

```text
32
```

Decision: use markdown on stdout for immediate reading, and `--output json` when a later command or script needs exact fields.

## Recommended Agent Flow

1. Start with `overview --turns N`.
2. Use `usage --output json | jq ...` to select costly turns from accounting data.
3. Use `step-detail` for readable evidence.
4. Use `event-detail` only when exact raw content or a scriptable field is needed.
